# 约束满足问题（CSP）― 来决定吃完便当的顺序

## 本章要点

本章讨论**约束满足问题**（Constraint Satisfaction Problem，简称 **CSP**）。

- 什么是约束满足问题
- 为什么穷举（枚举所有情况）很快就应付不了
- 为什么用 Z3 就能轻松求解

出现的关键词只有两个：“变量”和“约束”。请放松心情读下去。

## 什么是约束满足问题

约束满足问题，就是**给若干个变量定值，使它们满足给定条件（约束）的问题**。

先看一个小例子。

> 5 个小朋友（A, B, C, D, E）按先后顺序吃完了便当。
> 请根据下面的话，确定他们吃完的先后顺序。
>
> - A 比 B 吃得晚，但比 C 吃得早。
> - D 比 E 吃得早，但比 C 吃得晚。

在这个问题里，

- **变量**：5 个人各自“第几个吃完”
- **约束**：“A 在 B 之后”“A 在 C 之前”等条件

只要定出 5 人的顺序，并且没有违反任何条件，那就是答案。

## 整理条件

用文字不太好想，我们改用符号写一遍。

设 $X \prec Y$ 表示“X 比 Y 先吃完”。

1. A 比 B 晚、比 C 早 → $B \prec A \prec C$
2. D 比 E 早、比 C 晚 → $C \prec D \prec E$

把两个接起来，就成了一条链 $B \prec A \prec C \prec D \prec E$。

也就是说，要求的是“满足这个顺序的排列”。

## 先试着穷举求解

最直白的办法是：**把所有可能的顺序都造出来，只留下符合条件的**。

5 个人的排列只有 $5! = 120$ 种。全部试一遍也只是一瞬间。

In [1]:
from itertools import permutations

kids = ["A", "B", "C", "D", "E"]

# order 是“按吃完顺序排列的小孩编号”
for order in permutations(range(len(kids))):
    rank = {kids[i]: pos for pos, i in enumerate(order)}   # 每个小孩的名次
    if rank["B"] < rank["A"] < rank["C"] and rank["C"] < rank["D"] < rank["E"]:
        print(" → ".join(kids[i] for i in order))

B → A → C → D → E


符合条件的顺序只有 **B → A → C → D → E** 这一种。

5 个人的话一瞬间就完了。那么人数增加后会怎样呢？排列的数量按 $n!$ 增长。

In [2]:
import math

for n in [5, 8, 10, 12]:
    print(f"{n:2d} 人: {math.factorial(n):,} 种")

 5 人: 120 种
 8 人: 40,320 种
10 人: 3,628,800 种
12 人: 479,001,600 种


12 个人大约 4 亿 8 千万种。就算每秒试 100 万种，也要 8 分钟。到了 20 个人（约 2.4×10¹⁸ 种），根本算不完。

“全部试一遍”这种方法，人数稍微一多就不能用了。本章后半部分会看到 Z3 怎么跨过这堵墙。

### 补充：关于 `itertools.permutations`

穷举时用的 `permutations()`，是生成给定元素**所有排列**的函数。

```python
itertools.permutations(iterable, r=None)
```

- `iterable`：要排列的元素
- `r`：取几个（省略则全部）

3 个元素的话就是 $3! = 6$ 种。

In [3]:
for p in permutations(["A", "B", "C"]):
    print(p)

('A', 'B', 'C')
('A', 'C', 'B')
('B', 'A', 'C')
('B', 'C', 'A')
('C', 'A', 'B')
('C', 'B', 'A')


使用 `permutations()` 时要注意三点。

1. **区分顺序**：(A, B) 和 (B, A) 被当作不同的东西。
2. **不考虑重复**：元素有重复时，同样的排列会反复出现。
3. **计算量是 $n!$**：元素一多就爆炸式增长。

用来肉眼确认小问题很方便，但不适合大问题。这时就该求解器出场了。

## 用 Z3 求解

Z3 是**约束求解器**之一。只要把条件照原样写下来，它就会去找满足条件的取值组合。

与穷举的区别是这样。

| | 要做的事 |
|---|---|
| 穷举 | 把所有候补都造出来，一个一个检查 |
| Z3 | 只写条件，寻找交给 Z3 |

用 Z3 解题的步骤总是一样的。

1. 创建变量
2. 添加约束
3. 取出解

In [4]:
from z3 import IntVector, Solver, Distinct, sat

kids = ["A", "B", "C", "D", "E"]

# 表示每个小孩“吃完的名次”的 5 个整数变量（pos[0] 是 A，pos[1] 是 B …）
pos = IntVector("pos", 5)
eat_a, eat_b, eat_c, eat_d, eat_e = pos

In [5]:
solver = Solver()

# 1. 名次是 0 到 4 中的一个
for p in pos:
    solver.add(p >= 0, p < len(kids))

# 2. 没有名次相同的小孩
solver.add(Distinct(pos))

# 3. 问题的条件  B ≺ A ≺ C ≺ D ≺ E
solver.add(eat_b < eat_a, eat_a < eat_c)
solver.add(eat_c < eat_d, eat_d < eat_e)

# 4. 取出解，按名次从小到大排列
if solver.check() == sat:
    model = solver.model()
    rank = {kid: model[pos[i]].as_long() for i, kid in enumerate(kids)}
    print(" → ".join(sorted(kids, key=lambda kid: rank[kid])))

B → A → C → D → E


得到了和穷举相同的答案。请注意，能搜索的范围在一点点变小。

1. 只创建 `IntVector("pos", 5)`，值可以是任意 → 无限
2. 加上 `0 <= p < 5` → $5^5 = 3125$ 种
3. 加上 `Distinct(pos)` → $5! = 120$ 种（与穷举相同的范围）
4. 加上条件 $B \prec A \prec C \prec D \prec E$ → 1 种

穷举是从第 3 步的范围开始，自己一路试到最后。Z3 先接受条件，所以从一开始就不去造不符合条件的候补。人数增加，机制也一样。

## 小结

| 手法 | 计算量 | 适合的场景 |
|---|---|---|
| `itertools.permutations` | $O(n!)$ | 肉眼确认小问题 |
| Z3 | 由条件出发高效搜索 | 条件复杂的问题、人数多的问题 |

- 约束满足问题由“变量”和“约束”构成
- 穷举的计算量是 $n!$，问题稍微变大就不能用了
- Z3 可以把约束照原样写下来，问题变了也容易应付